# 02 - JSON para Delta Lake (MinIO - Camada Bronze)
 
Lê os arquivos JSON do bucket `landing-zone` no MinIO e grava cada um como tabela Delta Lake no bucket `bronze`.
 
**Pré-requisitos:** Notebook `01` executado (Arquivos JSON no MinIO).

## 1. Imports e Configuração

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

# Variáveis do MinIO
MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

MinIO: http://localhost:9020
Landing: landing-zone | Bronze: bronze


## 2. Criar SparkSession com Delta Lake e MinIO

In [2]:
spark = (
    SparkSession.builder
    .appName('JSON_to_Delta_Bronze')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

26/04/29 23:10:09 WARN Utils: Your hostname, joao-miguel resolves to a loopback address: 127.0.1.1; using 192.168.1.13 instead (on interface wlp0s20f3)
26/04/29 23:10:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/miguel/.ivy2/cache
The jars for the packages stored in: /home/miguel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-87b0bf57-1868-4564-a012-d86487a71d6f;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/miguel/trabalho-eng-dados-spark-delta-minio-mongo/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (561ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (106306ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-openssl-1.0.7.Final.jar ...
	[SUCCESSFUL ] org.wildfly.openssl#wildfly-openssl;1.0.7.Final!wildfly-openssl.jar (260ms)
:: resolution report :: resolve 6162ms :: artifacts dl 107135ms


SparkSession criada com sucesso!


## 3. Criar Bucket Bronze no MinIO

In [3]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] já existe.')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado com sucesso!')

print('Buckets disponíveis:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] criado com sucesso!
Buckets disponíveis: ['bronze', 'landing-zone']


## 4. Listar JSONs Disponíveis no Landing Zone

In [4]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
json_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]

print(f'{len(json_files)} arquivos JSON encontrados no bucket [{LANDING_BUCKET}]:')
for f in json_files:
    print(f'  - {f}')

4 arquivos JSON encontrados no bucket [landing-zone]:
  - ar_condicionado.json
  - clientes.json
  - vendas.json
  - vendedores.json


## 5. Ler JSONs e Gravar como Delta Lake

In [5]:
from delta.tables import DeltaTable

print(f'\nConvertendo {len(json_files)} JSONs para Delta Lake...\n')

for json_file in json_files:
    tabela = json_file.replace('.json', '')
    json_path = f's3a://{LANDING_BUCKET}/{json_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # 1. Ler o JSON.
    # Usamos multiline='true' porque o json.dumps gerou um array de objetos JSON de uma vez.
    df = spark.read \
        .option('multiline', 'true') \
        .json(json_path)
    
    # 2. Gravar como tabela Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)
    
    print(f'  ✅ {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversão concluída! {len(json_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')


Convertendo 4 JSONs para Delta Lake...



26/04/29 23:13:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


  ✅ ar_condicionado: 3 registros | 8 colunas -> s3a://bronze/ar_condicionado
  ✅ clientes: 2 registros | 4 colunas -> s3a://bronze/clientes
  ✅ vendas: 2 registros | 7 colunas -> s3a://bronze/vendas
  ✅ vendedores: 2 registros | 4 colunas -> s3a://bronze/vendedores

Conversão concluída! 4 tabelas Delta criadas no bucket [bronze].


## 6. Validação - Ler Tabelas Delta

In [6]:
print('Validando tabelas Delta Lake no bucket Bronze...\n')

for json_file in json_files:
    tabela = json_file.replace('.json', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Verificar se o formato Delta foi aplicado corretamente
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)
    
    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

Validando tabelas Delta Lake no bucket Bronze...



26/04/29 23:13:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  ar_condicionado: Delta=True | 3 registros | Colunas: ['_id', 'ano', 'fl_inverter', 'id', 'nm_marca', 'nm_modelo', 'potencia', 'valor']
  clientes: Delta=True | 2 registros | Colunas: ['_id', 'cd_cliente', 'cidade', 'nome']
  vendas: Delta=True | 2 registros | Colunas: ['_id', 'cd_cliente', 'cd_vendedor', 'data', 'id_produto', 'id_venda', 'quantidade']
  vendedores: Delta=True | 2 registros | Colunas: ['_id', 'cd_vendedor', 'nome', 'unidade']


In [7]:
# Amostra: exibir primeiros registros de ar condicionado
tabelas_amostra = ['ar_condicionado', 'clientes', 'vendedores', 'vendas']

for tabela in tabelas_amostra:
    # Verificamos se a tabela realmente está na lista de arquivos processados antes de ler
    if f"{tabela}.json" in json_files:
        print(f'\n--- {tabela.upper()} ---')
        spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)


--- AR_CONDICIONADO ---
+--------------------+----+-----------+---+--------+---------+---------+------+
|                 _id| ano|fl_inverter| id|nm_marca|nm_modelo| potencia| valor|
+--------------------+----+-----------+---+--------+---------+---------+------+
|69f2b9bf94a3b6945...|2024|       true|  1|      LG|Split 12k|12000 BTU|2500.0|
|69f2b9bf94a3b6945...|2023|      false|  2|  Consul|Janela 9k| 9000 BTU|1800.0|
|69f2b9bf94a3b6945...|2024|       true|  3| Samsung| WindFree|18000 BTU|3200.0|
+--------------------+----+-----------+---+--------+---------+---------+------+


--- CLIENTES ---
+--------------------+----------+--------------+------------+
|                 _id|cd_cliente|        cidade|        nome|
+--------------------+----------+--------------+------------+
|69f2b9bf94a3b6945...|       101|     São Paulo|Miguel Silva|
|69f2b9bf94a3b6945...|       102|Rio de Janeiro|Ana Oliveira|
+--------------------+----------+--------------+------------+


--- VENDEDORES ---
+--

In [8]:
spark.stop()
print('\nSparkSession finalizada.')


SparkSession finalizada.
